In [104]:
input_str = """nop +0
acc +1
jmp +4
acc +3
jmp -3
acc -99
acc +1
jmp -4
acc +6"""

In [115]:
with open('input_day8.txt', 'r') as f:
    input_str = f.read()

In [116]:
commands = (cmd.strip() for cmd in input_str.splitlines())
commands = (tuple(cmd.split()) for cmd in commands if cmd)
commands = [(cmd[0], int(cmd[1])) for cmd in commands]
#print(commands)

In [117]:
class Handheld:
    def __init__(self):
        self.instructions = {
            "nop": self.next_instruction,
            "acc": self.exec_acc,
            "jmp": self.exec_jmp,
        }
        
    def next_instruction(self, arg):
        self.instruction_counter += 1
    
    def exec_acc(self, arg):
        self.acc += arg
        self.next_instruction(arg)
        
    def exec_jmp(self, arg):
        self.instruction_counter += arg
    
    def run_program(self, program):
        self.acc = 0
        self.instruction_counter = 0
        
        visited_instructions = set()
        
        while True:
            #print(self.acc, self.instruction_counter)
            # check if the boot loader finished successfully (at instruction after bootloader)
            if self.instruction_counter == len(program):
                return True
            
            # check that the instruction counter is valid
            if self.instruction_counter not in range(len(program)):
                raise RuntimeError("Tried to access invalid operation")
                
            # check if infinite loop -> stop because invalid program
            if self.instruction_counter in visited_instructions:
                return False
            visited_instructions.add(self.instruction_counter)
            # fetch next instruction
            cmd, arg = program[self.instruction_counter]
            self.instructions[cmd](arg)
            
handheld = Handheld()

In [118]:
success = handheld.run_program(commands)
print(f"success: {success}, acc = {handheld.acc}")

success: False, acc = 1475


In [119]:
program = commands
for i in range(len(program)):
    cmd, arg = program[i]
    if cmd == "nop":
        altered_cmd = "jmp"
    elif cmd == "jmp":
        altered_cmd = "nop"
    else:
        continue # already tried this (original) combo out before unsuccessfully
    
    altered_program = program.copy()
    altered_program[i] = (altered_cmd, arg)
    success = handheld.run_program(altered_program)
    #print(f"Success: {success}, program: '{altered_program}'\n")
    if success:
        print(f"Successfully executed the program after altering line {i} from "
              f"'{program[i]}' to '{altered_program[i]}'")
        break
else:
    raise RuntimeError("No command swap lead to a successful execution of the program!")
print(handheld.acc)

Successfully executed the program after altering line 246 from '('jmp', 357)' to '('nop', 357)'
1270
